# 09.4 — Parameter sweeps

To find the best decoder — or to understand how sensitive it is to each choice — you *sweep*: systematically vary hyperparameters and compare results. The MATLAB harness enumerates **47 sweep variables** (architecture, learning rate, hidden sizes, augmentation strengths, …). But 47 variables can't be swept exhaustively — the cartesian product is astronomical. So the pipeline uses a **curated** sweep: a hand-authored table of 147 meaningful points, each varying *one* aspect from the Optimal baseline, addressed by the single `--sweep-index` ([09.2](09.2_slurm_dispatch.ipynb)). This notebook is that table and the coverage audit behind it.

This notebook covers:

* Why sweep, and why you can't sweep 47 variables exhaustively.
* The curated table: 147 points grouped by what they vary (`iter_by_choice`).
* One-variable-at-a-time sensitivity around the Optimal baseline.
* The 47-variable coverage matrix — honest port-completeness tracking.

**Prerequisites:** [09.2 SLURM dispatch](09.2_slurm_dispatch.ipynb), [09.3 config composition](09.3_hydra_config_composition.ipynb).

## Section 1 — What MATLAB does

`SLURMPARAMETERS_cgg_runAutoEncoder_v2.m` (lines 42-56) lists 47 variables the sweep can vary, addressed by two numbers — `SLURMChoice` (which variable group) and `SLURMIDX` (which value):

```matlab
% SLURMChoice picks WHAT to vary; SLURMIDX picks WHICH value
switch SLURMChoice
    case 1                          % vary the architecture
        cfg.ModelName = modelList{SLURMIDX};
    case 2                          % vary the L2 factor
        cfg.L2Factor = l2List{SLURMIDX};
    % ... 15 groups total
end
```

Each `SLURMChoice` block varies one aspect while holding the rest at the Optimal defaults. The port flattens `(SLURMChoice, SLURMIDX)` into the single sweep index ([09.2 §2.2](09.2_slurm_dispatch.ipynb)) and stores the whole table as `dispatcher.SWEEP_ENTRIES` — 147 named points.

## Section 2 — The Python concepts you need

### 2.1 — Why sweep

A model has many knobs: which architecture, how big the hidden layers, what learning rate, how much augmentation. You rarely know the best setting a priori. A **sweep** trains the model across a range of settings and compares — either to *find* the best (tuning) or to *measure* how much each knob matters (sensitivity analysis). It's the empirical backbone of "which config should we ship?"

### 2.2 — Why not sweep everything (the combinatorial wall)

In [ ]:
# 47 variables, even at just a few values each, explode combinatorially:
values_per_variable = 5
num_variables = 47
cartesian = values_per_variable ** num_variables
print(f"cartesian product of 47 variables @ {values_per_variable} values each:")
print(f"   {values_per_variable}^{num_variables} = {cartesian:.2e} combinations")
print(f"   at 1 minute each, that's {cartesian / 60 / 24 / 365:.2e} YEARS of compute.")
print("→ exhaustive sweeping is impossible. You must CURATE which points to run.")

A full grid over 47 variables is `5^47 ≈ 7×10³²` combinations — more than atoms in a person. Exhaustive sweeping is a non-starter. The practical answer is *curation*: a human picks the points worth running. The dominant strategy here is **one-variable-at-a-time from a strong baseline** — start from the Optimal config, and for each variable, try a handful of alternatives while holding everything else fixed. That measures each knob's effect without the combinatorial blowup.

### 2.3 — The curated table: 147 points in 15 groups

In [ ]:
from neural_data_decoding.sweeps import dispatcher

groups = list(dispatcher.iter_by_choice())
print(f"the sweep is {dispatcher.total_sweep_count()} points in {len(groups)} groups (one per SLURMChoice):")
for choice, entries in groups:
    print(f"  SLURMChoice {choice:2}: {len(entries):2} points — varies: {entries[0].description}")

Fifteen groups, 147 points total (14 groups of 10 + one of 7). Each group varies *one* dimension: architecture (SC1), L2 factor (SC2), data width (SC3), hidden sizes (SC4), classifier sizes (SC5), mini-batch size (SC6), and so on through augmentation (SC15). Within a group, the ~10 points are different *values* of that one variable. So the whole sweep is a structured sensitivity study around the Optimal baseline — deliberately *not* a blind grid.

### 2.4 — One variable at a time, live

In [ ]:
# Look at one group — how the architecture (SLURMChoice 1) is varied:
choice_1 = dict(groups)[1]
print("SLURMChoice 1 varies the MODEL ARCHITECTURE:")
for e in choice_1[:5]:
    model = e.overrides.get("model_name", "(baseline)")
    print(f"  sweep index {e.sweep_index:2}: {e.description:40} → model_name={model!r}")
print("  ...")
print()
# Each entry's overrides are SMALL — just the differences from Optimal (09.3):
e = dispatcher.lookup(21)
print(f"sweep index 21: {e.description}")
print(f"   overrides (only the differences): {e.overrides}")

Each entry's `overrides` dict is *small* — just the one-or-two keys that differ from the Optimal baseline ([09.3 §2.2](09.3_hydra_config_composition.ipynb)). Sweep index 21 might change only the classifier hidden sizes, leaving everything else at Optimal. This is what makes the sweep interpretable: because each point differs from the baseline in one controlled way, comparing its result to the baseline's *isolates that variable's effect*. It's a designed experiment, not a random search.

### 2.5 — The coverage matrix: honest port-completeness

In [ ]:
import neural_data_decoding.sweeps.parameter_coverage as coverage
print("parameter_coverage.py __all__:", coverage.__all__, "(docstring-only — an audit doc, no code)")
print()
# The module's docstring IS the 47-variable support matrix. A few rows:
rows = [ln for ln in coverage.__doc__.splitlines() if "``" in ln and ("✅" in ln or "◐" in ln or "N/A" in ln)]
for ln in rows[:6]:
    print("  ", " ".join(ln.split()))
print("   ... (the matrix documents all 47 sweep variables, one row each)")
print("\n✅ = supported · ◐ = partial (with a note) · N/A = not applicable to the Python pipeline")

A port is a *work in progress*, and honesty about *what's done* matters. `parameter_coverage.py` is a docstring-only module (`__all__ = []`, no executable API) whose entire content is a table auditing all 47 sweep variables: which are fully supported (`✅`), which are partial (`◐`, with a note explaining the gap — e.g. a field that passes through config but is only consumed by the real-data loader), and which are N/A to the Python pipeline. It's the same "registered ≠ implemented" honesty as [03.5](../03_data_pipeline/03.5_normalization_strategies.ipynb) and [08.3](../08_output_and_analysis/08.3_monitor_table_compatibility.ipynb), but at the *sweep* level: the matrix tells you exactly which sweep points are safe to run and which touch not-yet-complete paths. The parametrized integration tests it points to verify representative slices train end-to-end without crashing (a non-crash gate, no parity claim).

## Section 3 — The `neural_data_decoding` implementation

`SweepEntry` and the table it lives in:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/sweeps/dispatcher.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "class SweepEntry" in line)
j0 = next(j for j in range(i, len(src)) if src[j].strip().startswith("sweep_index:"))
for k in range(j0, j0 + 6):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `SweepEntry` carries `sweep_index` (the flat 1-based id), `matlab_choice`/`matlab_idx` (the original coordinates, for traceability), `description` (human label), `overrides` (the config diff), and `notes` (caveats, e.g. partial-support warnings).
* The 147 entries are a *hand-authored tuple* (`SWEEP_ENTRIES`), built at import — not generated from a cartesian product (§2.2). Curation is explicit and reviewable.
* `_translate_overrides` maps MATLAB CamelCase keys to the snake_case config keys ([09.3](09.3_hydra_config_composition.ipynb)) and *raises at import* on any unmapped key — so the table can't reference a config field the port doesn't know.
* `iter_by_choice()` regroups the flat table by `matlab_choice` (§2.3); `lookup(i)` / `lookup_by_choice(c, idx)` address a point either way — the flat index for the cluster, the MATLAB coordinates for cross-referencing the original sweep.
* Some indices are intentional duplicates or base-fillers preserved to keep the flat index aligned with MATLAB's — parity of *addressing*, so a given index means the same run in both systems.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the sweep is a sensitivity study

Show that most entries change only one or two config keys (they vary one thing from Optimal), by tallying override sizes across the whole table.

In [ ]:
# Reveal:
from collections import Counter
sizes = Counter(len(dispatcher.lookup(i).overrides) for i in range(1, dispatcher.total_sweep_count() + 1))
for n_keys in sorted(sizes):
    print(f"  {sizes[n_keys]:3} entries change {n_keys} config key(s)")
print("→ most points differ from Optimal in just a few keys — one-variable-at-a-time.")

### Exercise 4.2 — traceability both ways

Show that a flat sweep index and its MATLAB `(choice, idx)` coordinates address the *same* entry — so a Python job log and a MATLAB sweep row can be cross-referenced.

In [ ]:
# Reveal:
e = dispatcher.lookup(15)
same = dispatcher.lookup_by_choice(e.matlab_choice, e.matlab_idx)
print(f"flat index 15 → SC{e.matlab_choice}/IDX{e.matlab_idx}: {e.description}")
print(f"lookup_by_choice({e.matlab_choice}, {e.matlab_idx}) → index {same.sweep_index}: same entry? {same.sweep_index == 15}")
print("→ the two addressing schemes agree — full traceability to the MATLAB sweep table.")

## Section 5 — Common errors

### Trying to grid-search all 47 variables

Impossible (§2.2). Use the curated table — or, if you need a new combination, *add a `SweepEntry`* for it rather than generating a product. Curation is the point.

### A sweep point touches a partially-supported variable

Check the coverage matrix (§2.5) — a `◐` variable may not do what MATLAB does for all values. The `SweepEntry.notes` field flags many of these; the integration tests only assert non-crash, not parity, for those.

### The sweep index means a different run than expected

Flat indices are pinned to MATLAB's `(SLURMChoice, SLURMIDX)` ordering (§3), including deliberate duplicates. Don't renumber — cross-reference with `lookup_by_choice` to confirm which run an index is.

### Comparing sweep points that differ in more than one thing

Then you can't attribute a result to a single variable. The curated table keeps most points one-variable-from-Optimal for exactly this reason (§2.4); if you author a multi-change entry, you lose the clean sensitivity interpretation.

### An override key in a new entry isn't recognized

`_translate_overrides` raises at import on an unmapped MATLAB key (§3). Add the key to the translation map (or use the snake_case config key directly) so the table stays consistent with the config schema ([09.3](09.3_hydra_config_composition.ipynb)).

## Section 6 — Further reading

- [`src/neural_data_decoding/sweeps/dispatcher.py`](../../src/neural_data_decoding/sweeps/dispatcher.py) — the 147-entry `SWEEP_ENTRIES` table.
- [`src/neural_data_decoding/sweeps/parameter_coverage.py`](../../src/neural_data_decoding/sweeps/parameter_coverage.py) — the 47-variable coverage matrix.
- [`tests/integration/test_slurm_sweep_coverage.py`](../../tests/integration/test_slurm_sweep_coverage.py) — the non-crash gating tests.

Next up: **[09.5 debugging a failing run](09.5_debugging_a_failing_run.ipynb)** — a troubleshooting cookbook for when a sweep point goes wrong.